In [1]:
print('hello world ')

hello world 


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader

c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
loader=PyPDFLoader(r"../data/Paramétrage Etablissement FILIERIS.pdf")

In [17]:
loader

In [ ]:
dcouments=loader.load()

In [ ]:
dcouments

In [ ]:
len(dcouments)

In [ ]:
dcouments[0].metadata

In [ ]:
dcouments[22].page_content

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model_embedding = SentenceTransformer("BAAI/bge-m3")

In [ ]:
emb = model_embedding.encode([
    "Bonjour comment vas-tu ?",
    "Salut, ça va ?"
])

print(emb.shape)

In [14]:
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings

#from sentence_transformers import SentenceTransformer
#import faiss

In [15]:
# Embedding wrapper for LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17406.89it/s]


In [ ]:
dimension = 1024

# chroma db

In [ ]:
from langchain.document_loaders import TextLoader

In [ ]:
loader

In [ ]:
len(dcouments)

In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Lire le PDF
reader = PdfReader(
    r"C:\Users\tchouarr\Desktop\copi\doc a lire\Paramétrage Etablissement FILIERIS.pdf"
)

# Extraire le texte
DOC_ = ""

for page in reader.pages:
    text = page.extract_text()
    
    if text:
        DOC_ += text

In [ ]:
len(DOC_)

In [ ]:
91956 / 1200

In [ ]:
DOC_

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=False,
)


In [ ]:
texts = text_splitter.create_documents([DOC_])

In [ ]:
texts

In [ ]:
len(texts)

In [ ]:
texts[87]

In [ ]:
persist_directory="embeddings_pdf_63pages"

In [ ]:
model_embedding

In [ ]:
embedding_model

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
chroma_vdb=Chroma.from_documents(documents=texts,
                      embedding=embedding_model,
                      persist_directory=persist_directory,
                      )

In [ ]:
chroma_vdb.persist()

In [ ]:
vdb=Chroma(persist_directory=persist_directory,
        embedding_function=embedding_model)

In [ ]:
retriever=vdb.as_retriever()

In [ ]:
retriever.invoke("MOTIFS D'HOSPITALISATION")

In [ ]:
retriever = vdb.as_retriever(search_kwargs={"k": 10})

docs = retriever.invoke("MOTIFS D'HOSPITALISATION")

print(docs[0].page_content)
### i a donné le bon contexte a envoyé au générateur par la suite 

In [ ]:
print(docs[5].page_content)

# generation

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [4]:
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
groq_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [7]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans les documents administratifs français.

Réponds uniquement à partir du contexte fourni.

Contexte:
{context}

Question:
{question}

Si l'information n'est pas présente dans le contexte,
réponds:
"Information non trouvée dans les documents."
""")

In [ ]:
chain = prompt | llm | StrOutputParser()

In [ ]:

question = "CREATION LABORATOIRE D'ANALYSES BIOLOGIQUES"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

In [ ]:
question = "Comment importer des actes csarr ? "

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

In [ ]:
retriever.invoke(question)


In [ ]:
question = "Comment donner accès à la caisse HM à un utilisateur ? "

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

## astra db avec plus de de documents

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [9]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [10]:
from langchain_astradb import AstraDBVectorStore
from dotenv import load_dotenv
import os

In [11]:
load_dotenv()

True

In [12]:
db_api_endpoint = os.getenv("ASTRA_DB_API_ENDPOINT")
db_application_token = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
db_keyspace = os.getenv("ASTRA_DB_KEYSPACE")


In [29]:
from pathlib import Path
from pypdf import PdfReader
from docx import Document

ModuleNotFoundError: No module named 'exceptions'

In [30]:
def lire_pdf(chemin_pdf):

    texte = ""

    reader = PdfReader(chemin_pdf)

    for page in reader.pages:

        contenu = page.extract_text()

        if contenu:
            texte += contenu + "\n"

    return texte


def lire_docx(chemin_docx):

    doc = Document(chemin_docx)

    texte = ""

    for paragraphe in doc.paragraphs:
        texte += paragraphe.text + "\n"

    return texte

In [31]:

DOSSIER = Path(r"C:\Users\tchouarr\Desktop\projets\data")



In [32]:
DOC_ = ""

# PDFs
for pdf in DOSSIER.glob("*.pdf"):

    print(f"PDF : {pdf.name}")

    DOC_ += lire_pdf(pdf)

# DOCX
for docx in DOSSIER.glob("*.doc"):

    print(f"DOCX : {docx.name}")

    DOC_ += lire_docx(docx)



PDF : Paramétrage Etablissement FILIERIS.pdf
PDF : ReleaseNotes_1.2601_00_HopitalManager.pdf
PDF : [GP] Paramétrage des tarifs par groupe de partage.pdf
PDF : [GU] Calcul des besoins V12.pdf
DOCX : Procédure de création PS_DSI.doc


NameError: name 'Document' is not defined

In [33]:
len(DOC_)

167059

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [35]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)


In [36]:
texts = text_splitter.create_documents([DOC_])

In [37]:
len(texts)

227

In [38]:
texts

[Document(metadata={}, page_content='d \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nPARAMETRAGE \nETABLISSEMENT \nMETHODOLOGIE DE PARAMETRAGE D’UN ETABLISSEMENT \nFILIERIS \n \n \n \n \n \n \nMars 2020 \n \nRéalisé par : Arnaud FOUCHER – Consultant fonctionnel médical'),
 Document(metadata={}, page_content="PARAMETRAGE ETABLISSEMENT – FILIERIS \nSOMMAIRE \n1 CREATION DE L'ETABLISSEMENT ............................................................................................................................................... 1 \n1.1 UM ........................................................................................................................................................................................................................................................... 1 \n1.2 UFH .........................................................................................................................

In [40]:
collection_name="filieris_rag"

In [41]:
vstore = AstraDBVectorStore(
            embedding= embedding_model,
            collection_name=collection_name,
            api_endpoint=db_api_endpoint,
            token=db_application_token,
            namespace=db_keyspace,
        )

In [42]:
inserted_ids = vstore.add_documents(texts)
print(f"Successfully inserted {len(inserted_ids)} documents into AstraDB.")
      

Successfully inserted 227 documents into AstraDB.


In [ ]:
vstore = AstraDBVectorStore(
                embedding= self.model_loader.load_embeddings(),
                collection_name=collection_name,
                api_endpoint=self.db_api_endpoint,
                token=self.db_application_token,
                namespace=self.db_keyspace,
            )

In [43]:
top_k = 10
retriever = vstore.as_retriever(search_kwargs={"k": top_k})
print("Retriever loaded successfully.")

Retriever loaded successfully.


In [44]:
output=retriever.invoke("comment créer un profil utilsateur ")

In [45]:
output

[Document(id='8db35e0e218844eca3bc56fec085e95f', metadata={}, page_content='5 \n   \n PARAMETRAGE ETABLISSEMENT – FILIERIS \nCONSOLE / ANNUAIRES / UTILISATEURS / UTILISATEURS  \n \nPenser à créer les comptes tests ; associer les profils aux comptes tests existants OU créer un compte test s’il n’existe pas.  \nAccès : CONSOLE / ANNUAIRE / UTLISATEURS / UTILISATEURS / + \nProfessionnel de santé : chercher celui qui correspond (le nom de famille est ABSENT)  \nNom : automatiquement repris \nIdentifiant : testxxxxx xxxx -> ensemble de caractères permettant d’identifier le profil à tester  \nAccessibilité : tous PS et toutes uF \nDate activation : aujourd’hui \n \nProfil : cliquer sur le + \nAjouter le profil correspondant ; exemple : FIL_Kinésithérapeute / Etablissement : FILIERIS TRONC COMMUN \nEtablissement de connexion = Etablissement de référence (sinon, nous ne pouvons pas nous connecter en base avec les comptes \ntests puisqu’ils ne sont pas identifiés dans le LDAP). \n \nEn cas de c

In [46]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans les documents administratifs français.

Réponds uniquement à partir du contexte fourni.

Contexte:
{context}

Question:
{question}

Si l'information n'est pas présente dans le contexte,
réponds:
"Information non trouvée dans les documents."
""")

In [47]:
chain = prompt | llm | StrOutputParser()


In [48]:
question = "comment créer un profil utilsateur ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Pour créer un profil utilisateur, il faut suivre les étapes suivantes :

1. Accéder à la console : CONSOLE / ANNUAIRES / UTILISATEURS / UTILISATEURS 
2. Cliquer sur le "+" pour créer un nouveau compte test
3. Rechercher le professionnel de santé correspondant (même si le nom de famille est absent)
4. Le nom sera automatiquement repris
5. Choisir un identifiant unique pour le profil à tester (par exemple : testxxxxx xxxx)
6. Définir l'accessibilité (tous PS et toutes UF)
7. Définir la date d'activation (aujourd'hui)
8. Cliquer sur le "+" pour ajouter le profil correspondant (par exemple : FIL_Kinésithérapeute)
9. Sélectionner l'établissement de connexion (qui doit être le même que l'établissement de référence)

Il est important de noter que si le compte test n'existe pas, il faut le créer avant de procéder à l'association du profil.


In [49]:
question = "Où se paramètrent les valeurs du groupe XAP ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Les valeurs du groupe XAP se paramètrent dans : 
Paramétrage > Paramètres généraux > Valeurs paramètres 
o Dans la liste gauche sélectionner le tiroir : 
o Paramètres par groupe 
▪ Puis sélectionner XAP


In [50]:

question = "À quoi sert le paramètre XAFFCHGCDE ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le paramètre XAFFCHGCDE sert à afficher ou non la fenêtre de saisie des paramètres en saisie de commande après avoir cliqué sur le bouton "Chargement".


In [ ]:

question = "Que fait le paramètre XENCCLIENT ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le paramètre XENCCLIENT permet de prendre en compte ou non les encours de commandes services lors du calcul des besoins.


In [50]:



question = "Quelle fonctionnalité liée aux FORMS n’est disponible qu’en prescription multimodale dans cette version ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

La fonctionnalité liée aux FORMS qui n’est disponible qu’en prescription multimodale dans cette version est la possibilité pour l’utilisateur d’associer un FORMS à un soin, de sorte à ce qu’à la prescription de ce soin le FORMS s’ouvre et le médecin peut saisir des informations complémentaires à destination du soignant. Elle sera libérée pour la prescription simplifiée et unitaire dans un patch 1.2601.xx à venir.


In [51]:




question = "Quel paramètre permet d’activer la latéralité structurée des antécédents ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le paramètre qui permet d'activer la latéralité structurée des antécédents est "ACTIV_LATERALITE_ANTECEDENT".


In [52]:

question = "Que permet spécifiquement l’option “Uniquement les perfusions” du paramètre ANESTHESIE_CONFIG ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

L'option "Uniquement les perfusions" du paramètre ANESTHESIE_CONFIG permet de stopper, renouveler, changer le débit ou ajouter un bolus sur les administrations de type perfusion/PSE/AIVOC ou Gaz saisies dans une phase antérieure. Les autres types d'administrations ou items ne seront pas modifiables.


In [53]:

question = "Quel module introduit une nouvelle habilitation RNIV liée à l’INSi ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le module GAP introduit une nouvelle habilitation RNIV liée à l'INSi, plus précisément dans la fonction "Patient" sous-onglet "Patient". Cette habilitation est : "[Appel INSI] – Ecran Patient sous-onglet Patient".


In [54]:


question = "Quelle correction concernait un problème de récupération du poids dans une prescription en /kg ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

La correction 602013 concerne un problème de récupération du poids dans une prescription en /kg lorsque le groupe de partage des constantes est actif. Cette correction a été appliquée dans la Release 1.2601.00 et concerne le Bloc module : Circuit du Patient Opéré fonction : Soins-critiques. Elle a été reportée dans les versions 1.2511.XX, 1.2509.XX et 1.2505.XX (numéros de report : 607260, 607259 et 607258).


In [55]:

question = "Quelle évolution modifie le comportement du blocage de conclusion des urgences ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

L'évolution 583991 modifie le comportement du blocage de conclusion des urgences. Elle permet désormais de paramétrer le type du contrôle via l'indicateur de blocage "CONCLURG – Conclusion du dossier des urgences manquante" et ses 2 natures "CCLURG" (Bloquant) et "WARCCLURG" (Warning).


In [51]:

question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin est effectué de la manière suivante :

1. **Comparaison seuil de réappro par rapport au stock de sécurité** : Si le stock de sécurité est supérieur au seuil de réappro, alors le seuil de réappro est égal au stock de sécurité.
2. **Calcul du disponible théorique** : Le disponible théorique est calculé en fonction de la formule suivante : Dispo = Stock physique - Alloué - Manquants + Encours fournisseurs - Encours services.
3. **Comparaison seuil de réappro par rapport au disponible théorique** : Si le seuil de réappro est supérieur au disponible théorique, alors le besoin brut est calculé en fonction de la consommation de la période explorée et du nombre de jours de consommation.
4. **Calcul du besoin net** : Le besoin net est calculé en fonction du besoin brut et du disponible théorique, en arrondissant le résultat au conditionnement supérieur ou en fonction de la QMA (Quantité minimum d'achat) de la fiche article-fournisseur.

Il est important de noter que le calcu

## faux 
p18 sage x3

In [52]:

question = "comment calculer le besoin d'un site ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin d'un site peut être effectué en suivant les étapes suivantes :

1. **Comparaison seuil de réappro par rapport au stock de sécurité** : Si le stock de sécurité saisi (dans la fiche article-site) est supérieur au seuil de réappro, alors le seuil de réappro est égal au stock de sécurité.
2. **Calcul du disponible théorique** : Le disponible théorique est calculé en fonction de la formule suivante : Dispo = Stock physique - Alloué - Manquants + Encours fournisseurs - Encours services.
3. **Comparaison seuil de réappro par rapport au disponible théorique** : Si le seuil de réappro est supérieur au disponible théorique, alors le besoin brut est calculé en fonction de la consommation de la période explorée et du nombre de jours de consommation pour le calcul du besoin (fonction du type de délai).
4. **Calcul du besoin net** : Le besoin net est calculé en soustrayant le disponible théorique du besoin brut. Le besoin net est ensuite arrondi au conditionnement supérieur ou en